In [1]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

# =========================
# INIT
# =========================
fake = Faker("en_IN")
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# =========================
# CONFIG
# =========================
NUM_CUSTOMERS = 8000
TARGET_ACCOUNTS = 10000
TARGET_TRANSACTIONS = 250000

TX_START = datetime(2024, 7, 1)
TX_END = datetime(2024, 12, 31)
ACC_OPEN_START = datetime(1950, 1, 1)
ACC_OPEN_END = datetime(2024, 12, 31)

# =========================
# 1. CUSTOMERS
# =========================
print("Generating CUSTOMERS...")

customer_ids = [f"CUST_{i:05d}" for i in range(1, NUM_CUSTOMERS + 1)]

df_customers = pd.DataFrame({
    "customer_id": customer_ids,
    "customer_type": np.random.choice(
        ["INDIVIDUAL", "CORPORATE", "NPO", "NRI"],
        NUM_CUSTOMERS,
        p=[0.85, 0.10, 0.03, 0.02]
    ),
    "gender": np.random.choice(["M", "F", "O"], NUM_CUSTOMERS, p=[0.52, 0.47, 0.01]),
    "income_bracket": np.random.choice(
        ["<5L", "5-10L", "10-25L", ">25L"],
        NUM_CUSTOMERS,
        p=[0.60, 0.25, 0.10, 0.05]
    ),
    "kyc_status": np.random.choice(
        ["KYC_COMPLETE", "PENDING", "EXPIRED"],
        NUM_CUSTOMERS,
        p=[0.90, 0.08, 0.02]
    ),
    "pep_flag": np.random.choice(["Y", "N"], NUM_CUSTOMERS, p=[0.01, 0.99]),
    "sanction_flag": np.random.choice(["Y", "N"], NUM_CUSTOMERS, p=[0.005, 0.995]),
    "internal_watchlist_flag": np.random.choice(["Y", "N"], NUM_CUSTOMERS, p=[0.02, 0.98]),
    "customer_risk_rating": np.random.choice(
        ["LOW", "MEDIUM", "HIGH"],
        NUM_CUSTOMERS,
        p=[0.70, 0.25, 0.05]
    ),
    "dob_or_incorporation_date": [
        fake.date_of_birth(minimum_age=18, maximum_age=90)
        for _ in range(NUM_CUSTOMERS)
    ]
})

df_customers["customer_segment"] = df_customers["customer_type"].apply(
    lambda x: "CORPORATE" if x == "CORPORATE" else "RETAIL"
)

# =========================
# 2. ACCOUNTS (STRICT 1–3)
# =========================
print("Generating ACCOUNTS...")

accounts = []
acc_counter = 1

for _, cust in df_customers.iterrows():
    num_accounts = np.random.choice([1, 2, 3], p=[0.80, 0.15, 0.05])

    for _ in range(num_accounts):
        if acc_counter > TARGET_ACCOUNTS:
            break

        open_date = fake.date_between(ACC_OPEN_START, ACC_OPEN_END)
        status = np.random.choice(["ACTIVE", "DORMANT", "INACTIVE"], p=[0.85, 0.10, 0.05])

        last_dormant_date = None
        if status == "DORMANT":
            last_dormant_date = fake.date_between(open_date, TX_END)

        close_date = None
        if status == "INACTIVE":
            close_date = fake.date_between(open_date, TX_END)

        accounts.append({
            "account_id": f"ACC_{acc_counter:06d}",
            "customer_id": cust["customer_id"],
            "account_type": (
                "CURRENT" if cust["customer_type"] == "CORPORATE"
                else np.random.choice(["SAVINGS", "CURRENT"], p=[0.9, 0.1])
            ),
            "account_open_date": open_date,
            "account_status": status,
            "dormancy_flag": "Y" if status == "DORMANT" else "N",
            "last_dormant_date": last_dormant_date,
            "account_close_date": close_date
        })

        acc_counter += 1

df_accounts = pd.DataFrame(accounts)

# =========================
# 3. TRANSACTIONS (FAST + VALID)
# =========================
print("Generating TRANSACTIONS...")

valid_accounts = []

for _, acc in df_accounts.iterrows():
    start = max(TX_START.date(), acc["account_open_date"])
    end = TX_END.date()

    if pd.notna(acc["account_close_date"]):
        end = min(end, acc["account_close_date"])

    if acc["dormancy_flag"] == "Y" and pd.notna(acc["last_dormant_date"]):
        end = min(end, acc["last_dormant_date"] - timedelta(days=180))

    if start <= end:
        valid_accounts.append((acc["account_id"], acc["customer_id"], start, end))

tx_rows = []
tx_categories = ["NEFT", "RTGS", "IMPS", "UPI", "CHEQUE", "INTERNATIONAL_WIRE", "FX"]

for i in range(TARGET_TRANSACTIONS):
    acc_id, cust_id, start, end = random.choice(valid_accounts)
    tx_date = start + timedelta(days=random.randint(0, (end - start).days))

    is_cash = random.random() < 0.05

    tx_rows.append({
        "transaction_id": f"TXN_{i:08d}",
        "account_id": acc_id,
        "customer_id": cust_id,
        "transaction_datetime": tx_date,
        "transaction_type": random.choice(["DEBIT", "CREDIT"]),
        "transaction_category": "CASH" if is_cash else random.choice(tx_categories),
        "transaction_amount": round(
            random.uniform(1000, 50000) if is_cash else random.uniform(100, 5_000_000), 2
        )
    })

df_transactions = pd.DataFrame(tx_rows)

# =========================
# 4. EXPORT (STRICT)
# =========================
print("Exporting CSVs...")

# df_customers.to_csv(r"E:\VS code stuff\Banks Data\DB_bank\customers.csv", index=False)
# df_accounts.to_csv(r"E:\VS code stuff\Banks Data\DB_bank\accounts.csv", index=False)
# df_transactions.to_csv(r"E:\VS code stuff\Banks Data\DB_bank\transactions.csv", index=False)

print("DONE")
print("Customers:", len(df_customers))
print("Accounts:", len(df_accounts))
print("Transactions:", len(df_transactions))


Generating CUSTOMERS...
Generating ACCOUNTS...
Generating TRANSACTIONS...
Exporting CSVs...
DONE
Customers: 8000
Accounts: 9979
Transactions: 250000


In [5]:
import pandas as pd
df_customers = pd.read_csv(r"E:\VS code stuff\Banks Data\DB_bank\customers.csv")
df_accounts = pd.read_csv(r"E:\VS code stuff\Banks Data\DB_bank\accounts.csv")
df_transactions = pd.read_csv(r"E:\VS code stuff\Banks Data\DB_bank\transactions.csv")
df_strs = pd.read_csv(r"E:\VS code stuff\Banks Data\DB_bank\str.csv")

In [ ]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

# =========================
# INIT
# =========================
fake = Faker("en_IN")
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# =========================
# CONFIG
# =========================
NUM_CUSTOMERS = 8000
TARGET_ACCOUNTS = 10000
TARGET_TRANSACTIONS = 250000

TX_START = datetime(2024, 7, 1)
TX_END = datetime(2024, 12, 31)
ACC_OPEN_START = datetime(1950, 1, 1)
ACC_OPEN_END = datetime(2024, 12, 31)

# =========================
# 1. CUSTOMERS
# =========================
print("Generating CUSTOMERS...")

customer_ids = [f"CUST_{i:05d}" for i in range(1, NUM_CUSTOMERS + 1)]

df_customers = pd.DataFrame({
    "customer_id": customer_ids,
    "customer_type": np.random.choice(
        ["INDIVIDUAL", "CORPORATE", "NPO", "NRI"],
        NUM_CUSTOMERS,
        p=[0.85, 0.10, 0.03, 0.02]
    ),
    "gender": np.random.choice(["M", "F", "O"], NUM_CUSTOMERS, p=[0.52, 0.47, 0.01]),
    "income_bracket": np.random.choice(
        ["<5L", "5-10L", "10-25L", ">25L"],
        NUM_CUSTOMERS,
        p=[0.60, 0.25, 0.10, 0.05]
    ),
    "kyc_status": np.random.choice(
        ["KYC_COMPLETE", "PENDING", "EXPIRED"],
        NUM_CUSTOMERS,
        p=[0.90, 0.08, 0.02]
    ),
    "pep_flag": np.random.choice(["Y", "N"], NUM_CUSTOMERS, p=[0.01, 0.99]),
    "sanction_flag": np.random.choice(["Y", "N"], NUM_CUSTOMERS, p=[0.005, 0.995]),
    "internal_watchlist_flag": np.random.choice(["Y", "N"], NUM_CUSTOMERS, p=[0.02, 0.98]),
    "customer_risk_rating": np.random.choice(
        ["LOW", "MEDIUM", "HIGH"],
        NUM_CUSTOMERS,
        p=[0.70, 0.25, 0.05]
    ),
    "dob_or_incorporation_date": [
        fake.date_of_birth(minimum_age=18, maximum_age=90)
        for _ in range(NUM_CUSTOMERS)
    ]
})

df_customers["customer_segment"] = np.where(
    df_customers["customer_type"] == "CORPORATE",
    "CORPORATE",
    "RETAIL"
)

# =========================
# 2. ACCOUNTS
# =========================
print("Generating ACCOUNTS...")

accounts = []
acc_counter = 1

for _, cust in df_customers.iterrows():
    num_accounts = np.random.choice([1, 2, 3], p=[0.80, 0.15, 0.05])

    for _ in range(num_accounts):
        if acc_counter > TARGET_ACCOUNTS:
            break

        open_date = fake.date_between(ACC_OPEN_START, ACC_OPEN_END)
        status = np.random.choice(["ACTIVE", "DORMANT", "INACTIVE"], p=[0.85, 0.10, 0.05])

        last_dormant_date = None
        if status == "DORMANT":
            last_dormant_date = fake.date_between(open_date, TX_END)

        close_date = None
        if status == "INACTIVE":
            close_date = fake.date_between(open_date, TX_END)

        accounts.append({
            "account_id": f"ACC_{acc_counter:06d}",
            "customer_id": cust["customer_id"],
            "account_type": (
                "CURRENT" if cust["customer_type"] == "CORPORATE"
                else np.random.choice(["SAVINGS", "CURRENT"], p=[0.9, 0.1])
            ),
            "account_open_date": open_date,
            "account_status": status,
            "dormancy_flag": "Y" if status == "DORMANT" else "N",
            "last_dormant_date": last_dormant_date,
            "account_close_date": close_date
        })

        acc_counter += 1

df_accounts = pd.DataFrame(accounts)

# =========================
# 3. TRANSACTIONS
# =========================
print("Generating TRANSACTIONS...")

valid_accounts = []

for _, acc in df_accounts.iterrows():
    start = max(TX_START.date(), acc["account_open_date"])
    end = TX_END.date()

    if pd.notna(acc["account_close_date"]):
        end = min(end, acc["account_close_date"])

    if acc["dormancy_flag"] == "Y" and pd.notna(acc["last_dormant_date"]):
        end = min(end, acc["last_dormant_date"] - timedelta(days=180))

    if start <= end:
        valid_accounts.append((acc["account_id"], acc["customer_id"], start, end))

tx_rows = []
tx_categories = ["NEFT", "RTGS", "IMPS", "UPI", "CHEQUE", "INTERNATIONAL_WIRE", "FX"]

for i in range(TARGET_TRANSACTIONS):
    acc_id, cust_id, start, end = random.choice(valid_accounts)
    tx_date = start + timedelta(days=random.randint(0, (end - start).days))

    is_cash = random.random() < 0.05

    tx_rows.append({
        "transaction_id": f"TXN_{i:08d}",
        "account_id": acc_id,
        "customer_id": cust_id,
        "transaction_datetime": pd.to_datetime(tx_date),
        "transaction_type": random.choice(["DEBIT", "CREDIT"]),
        "transaction_category": "CASH" if is_cash else random.choice(tx_categories),
        "transaction_amount": round(
            random.uniform(1000, 50000) if is_cash else random.uniform(100, 5_000_000), 2
        )
    })

df_transactions = pd.DataFrame(tx_rows)

# =========================
# 4. STR GENERATION
# =========================
print("Generating STR data...")

NUM_STR = 300
MONTHLY_STR = 50

cash_tx = df_transactions[df_transactions["transaction_category"] == "CASH"].copy()

cash_tx = cash_tx.merge(
    df_customers[["customer_id", "customer_risk_rating"]],
    on="customer_id",
    how="left"
)

cash_tx["risk_weight"] = cash_tx["customer_risk_rating"].map(
    {"HIGH": 3, "MEDIUM": 2, "LOW": 1}
)

cash_tx = cash_tx.sort_values(
    ["risk_weight", "transaction_amount"],
    ascending=[False, False]
)

selected_accounts = set()
str_rows = []
str_counter = 1

monthly_groups = cash_tx.groupby(
    cash_tx["transaction_datetime"].dt.to_period("M")
)

for _, group in monthly_groups:
    if len(str_rows) >= NUM_STR:
        break

    group = group[~group["account_id"].isin(selected_accounts)]
    sampled = group.head(MONTHLY_STR)

    for _, tx in sampled.iterrows():
        tx_date = tx["transaction_datetime"]
        str_date = tx_date + timedelta(days=random.randint(1, 30))
        str_date = min(str_date, TX_END)

        str_rows.append({
            "account_id": tx["account_id"],
            "str_filed_date": str_date
        })

        selected_accounts.add(tx["account_id"])
        str_counter += 1

        if len(str_rows) >= NUM_STR:
            break

df_str = pd.DataFrame(str_rows)

print("STRs generated:", len(df_str))


Generating CUSTOMERS...
Generating ACCOUNTS...
Generating TRANSACTIONS...
Generating STR data...
STRs generated: 300


In [18]:
df_str.drop(columns=['str_id','reference_transaction_date'], inplace=True)

In [22]:
df_str.columns

Index(['account_id', 'str_filed_date'], dtype='object')

In [6]:
df_str["month"] = df_str["str_filed_date"].dt.to_period("M")
df_str["month"].value_counts()


month
2024-12    73
2024-10    56
2024-08    53
2024-11    51
2024-09    40
2024-07    27
Freq: M, Name: count, dtype: int64

In [21]:
df_str.columns

Index(['account_id', 'str_filed_date'], dtype='object')

In [20]:
df_str.to_csv(r"E:\VS code stuff\Banks Data\DB_bank\str.csv", index=False)

In [14]:
df_str

,str_id,account_id,reference_transaction_date,str_filed_date,month
0,STR_00001,ACC_005000,2024-07-16,2024-07-18,2024-07
1,STR_00002,ACC_004249,2024-07-29,2024-08-15,2024-08
2,STR_00003,ACC_008174,2024-07-24,2024-08-20,2024-08
3,STR_00004,ACC_008841,2024-07-10,2024-07-20,2024-07
4,STR_00005,ACC_000126,2024-07-14,2024-07-28,2024-07
...,...,...,...,...,...
295,STR_00296,ACC_008479,2024-12-14,2024-12-22,2024-12
296,STR_00297,ACC_004524,2024-12-19,2024-12-24,2024-12
297,STR_00298,ACC_008292,2024-12-17,2024-12-31,2024-12
298,STR_00299,ACC_001400,2024-12-26,2024-12-31,2024-12


In [8]:
(df_str["str_filed_date"] > df_str["reference_transaction_date"]).all()


False